In [ ]:
!nvidia-smi

In [ ]:
!pip install -q "transformers>=4.49.0" "peft>=0.13.0" "bitsandbytes>=0.44.0" \
  "datasets>=3.0.0" "accelerate>=1.0.0" "trl>=0.12.0" huggingface_hub pyyaml

In [ ]:
from pathlib import Path
from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
Path("/content/drive/MyDrive/ConvoBridge").mkdir(parents=True, exist_ok=True)
Path("/content/drive/MyDrive/ConvoBridge/chatbot_checkpoints").mkdir(parents=True, exist_ok=True)
print("Google Drive mounted.")

login()
# login(token="hf_xxxxxxxx")

In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> GPU"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()

GPU_NAME = torch.cuda.get_device_name(0)
GPU_TOTAL_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print(f"GPU: {GPU_NAME} | Total VRAM: {GPU_TOTAL_GB} GB")

BASE_MODEL = "google/gemma-3-1b-it"

DRIVE_ROOT = Path("/content/drive/MyDrive/ConvoBridge")
OUTPUT_DIR = DRIVE_ROOT / "gemma3-chatbot-lora"
CHECKPOINT_DIR = DRIVE_ROOT / "chatbot_checkpoints"

# FAST Colab profile (~45-90 min on T4)
# Old "max GPU" settings (batch=64, rows=8000, seq=2048, auto_find=True)
# made training take many hours and often hang on batch search.
TRAIN_CFG = {
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 2e-4,
    "num_epochs": 1,                 # 1 epoch is enough for Q&A LoRA warm-up
    "per_device_batch_size": 8,      # fixed batch (no slow auto-search)
    "gradient_accumulation_steps": 2, # effective batch = 16
    "max_seq_length": 1024,          # much faster than 2048
    "save_steps": 50,
    "max_train_rows": 2500,          # finish before Colab timeout
    "packing": False,               # packing can slow tokenization setup
    "auto_find_batch_size": False,  # CRITICAL: auto-find wastes a lot of time
}

PROMPT_TEMPLATE = """You are ConvoBridge Meeting Q&A Assistant.
Answer the user question using ONLY the meeting transcript context below.

Rules:
- Use only facts present in the transcript.
- If the answer is not in the transcript, reply exactly: Not mentioned in the transcript.
- Keep the answer short and clear (1-4 sentences).
- Keep the same language as the question when possible.

TRANSCRIPT:
{transcript}

QUESTION:
{question}"""

eff = TRAIN_CFG["per_device_batch_size"] * TRAIN_CFG["gradient_accumulation_steps"]
est_steps = (TRAIN_CFG["max_train_rows"] // max(eff, 1)) * TRAIN_CFG["num_epochs"]
print("Model:", BASE_MODEL)
print("FAST mode — Epochs:", TRAIN_CFG["num_epochs"],
      "| Rows:", TRAIN_CFG["max_train_rows"],
      "| Batch:", TRAIN_CFG["per_device_batch_size"],
      "| Seq:", TRAIN_CFG["max_seq_length"])
print("Estimated steps ~", est_steps, "(~45-90 min on T4)")
print("If CUDA OOM: set per_device_batch_size to 4")

In [ ]:
import re
import random
from datasets import load_dataset

random.seed(42)
UNKNOWN = "Not mentioned in the transcript."


def speakers_from_dialogue(dialogue: str) -> list[str]:
    names = []
    for line in dialogue.splitlines():
        if ":" in line:
            name = line.split(":", 1)[0].strip()
            if name and name not in names and len(name) < 30:
                names.append(name)
    return names


def line_for_speaker(dialogue: str, speaker: str) -> str | None:
    parts = []
    for line in dialogue.splitlines():
        if line.startswith(f"{speaker}:"):
            parts.append(line.split(":", 1)[1].strip())
    if not parts:
        return None
    return " ".join(parts)


def make_samsum_qa(dialogue: str, summary: str) -> list[dict]:
    """Create multiple Q&A rows from one dialogue."""
    dialogue = dialogue.strip()
    summary = summary.strip()
    if not dialogue or not summary:
        return []

    rows = []
    names = speakers_from_dialogue(dialogue)

    # Core meeting-style questions grounded in summary/dialogue
    rows.append(
        {
            "transcript": dialogue,
            "question": "What is this conversation mainly about?",
            "answer": summary,
            "source": "samsum",
        }
    )
    rows.append(
        {
            "transcript": dialogue,
            "question": "Summarize the key points briefly.",
            "answer": summary,
            "source": "samsum",
        }
    )

    for speaker in names[:3]:
        spoken = line_for_speaker(dialogue, speaker)
        if spoken:
            rows.append(
                {
                    "transcript": dialogue,
                    "question": f"What did {speaker} say?",
                    "answer": spoken,
                    "source": "samsum",
                }
            )
            rows.append(
                {
                    "transcript": dialogue,
                    "question": f"Who mentioned: {spoken[:80]}",
                    "answer": speaker,
                    "source": "samsum",
                }
            )

    # Negative / out-of-context question (important for grounded chatbot)
    fake_q = random.choice(
        [
            "What is the project budget in USD?",
            "When is the company IPO date?",
            "Who is the CEO of OpenAI?",
            "What is Ahmed's passport number?",
        ]
    )
    rows.append(
        {
            "transcript": dialogue,
            "question": fake_q,
            "answer": UNKNOWN,
            "source": "samsum_neg",
        }
    )
    return rows


def make_squad_qa(ex: dict) -> dict | None:
    context = (ex.get("context") or "").strip()
    question = (ex.get("question") or "").strip()
    answers = ex.get("answers") or {}
    texts = answers.get("text") or []
    if not context or not question or not texts:
        return None
    answer = texts[0].strip()
    if not answer:
        return None
    return {
        "transcript": context,
        "question": question,
        "answer": answer,
        "source": "squad",
    }


def load_online_qa_rows(max_rows: int) -> list[dict]:
    rows: list[dict] = []

    # 1) SAMSum dialogues -> meeting-style QA (~70% budget)
    samsum_budget = int(max_rows * 0.7)
    print("Loading knkarthick/samsum ...")
    samsum = load_dataset("knkarthick/samsum")["train"]
    for ex in samsum:
        for qa in make_samsum_qa(ex.get("dialogue", ""), ex.get("summary", "")):
            rows.append(qa)
            if len(rows) >= samsum_budget:
                break
        if len(rows) >= samsum_budget:
            break
    print(f"  SAMSum-derived rows: {len(rows)}")

    # 2) SQuAD for grounded extractive QA (~30%)
    print("Loading rajpurkar/squad ...")
    try:
        squad = load_dataset("rajpurkar/squad")["train"]
        added = 0
        for ex in squad:
            qa = make_squad_qa(ex)
            if not qa:
                continue
            # Keep shorter contexts for training speed
            if len(qa["transcript"].split()) > 350:
                continue
            rows.append(qa)
            added += 1
            if len(rows) >= max_rows:
                break
        print(f"  Added SQuAD rows: {added}")
    except Exception as exc:
        print(f"  SQuAD skipped: {exc}")

    random.shuffle(rows)
    rows = rows[:max_rows]
    print(f"Total Q&A training rows: {len(rows)}")
    return rows


train_rows = load_online_qa_rows(TRAIN_CFG["max_train_rows"])
assert len(train_rows) > 0, "No Q&A data loaded."
train_rows[0]

In [ ]:
from datasets import Dataset


def format_example(row: dict) -> str:
    user_content = PROMPT_TEMPLATE.format(
        transcript=row["transcript"],
        question=row["question"],
    )
    return (
        f"<start_of_turn>user\n{user_content}<end_of_turn>\n"
        f"<start_of_turn>model\n{row['answer']}<end_of_turn>"
    )


formatted = [format_example(r) for r in train_rows]
train_ds = Dataset.from_dict({"text": formatted})
print("Rows:", len(train_ds))
print(formatted[0][:1100], "...")

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

try:
    from trl import SFTConfig
except ImportError:
    SFTConfig = None

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# No max_memory / low_cpu_mem restrictions — full GPU allowed
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora_config = LoraConfig(
    r=TRAIN_CFG["lora_r"],
    lora_alpha=TRAIN_CFG["lora_alpha"],
    lora_dropout=TRAIN_CFG["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

resume_ckpt = None
ckpts = sorted(
    CHECKPOINT_DIR.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]),
)
if ckpts:
    resume_ckpt = str(ckpts[-1])
    print("Resuming from:", resume_ckpt)
else:
    print("No checkpoint found — starting fresh.")


def build_trainer(model, tokenizer, train_ds):
    common = dict(
        output_dir=str(CHECKPOINT_DIR),
        num_train_epochs=TRAIN_CFG["num_epochs"],
        per_device_train_batch_size=TRAIN_CFG["per_device_batch_size"],
        gradient_accumulation_steps=TRAIN_CFG["gradient_accumulation_steps"],
        learning_rate=TRAIN_CFG["learning_rate"],
        logging_steps=10,
        save_steps=TRAIN_CFG["save_steps"],
        save_total_limit=3,
        bf16=torch.cuda.is_available(),
        gradient_checkpointing=True,           # lower VRAM, more stable
        auto_find_batch_size=False,            # disabled — was extremely slow
        optim="adamw_torch",                   # avoid bitsandbytes 8bit CUDA crashes
        max_grad_norm=1.0,
        report_to="none",
        dataloader_pin_memory=False,
        dataloader_num_workers=0,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
    )

    # Prefer new TRL API; packing OFF by default for faster startup
    if SFTConfig is not None:
        for packing in (False, TRAIN_CFG.get("packing", False)):
            try:
                args = SFTConfig(
                    max_length=TRAIN_CFG["max_seq_length"],
                    packing=packing,
                    **common,
                )
                print(f"Using SFTConfig (packing={packing})")
                return SFTTrainer(
                    model=model,
                    args=args,
                    train_dataset=train_ds,
                    processing_class=tokenizer,
                )
            except TypeError:
                continue

    args = TrainingArguments(**common)
    try:
        return SFTTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            processing_class=tokenizer,
            max_seq_length=TRAIN_CFG["max_seq_length"],
        )
    except TypeError:
        return SFTTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            tokenizer=tokenizer,
            max_seq_length=TRAIN_CFG["max_seq_length"],
        )


trainer = build_trainer(model, tokenizer, train_ds)

print("GPU total:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
print("GPU allocated before train:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
print("GPU reserved before train:", round(torch.cuda.memory_reserved() / 1e9, 2), "GB")
print("Starting train — expect ~45-90 minutes on T4…")

trainer.train(resume_from_checkpoint=resume_ckpt)

trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved chatbot LoRA adapter ->", OUTPUT_DIR)
print("GPU peak reserved:", round(torch.cuda.max_memory_reserved() / 1e9, 2), "GB")
print(
    "Peak utilization ~",
    round(
        100 * torch.cuda.max_memory_reserved()
        / torch.cuda.get_device_properties(0).total_memory,
        1,
    ),
    "% of GPU VRAM",
)

In [ ]:
model.eval()
model.config.use_cache = True
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()


def ask(transcript: str, question: str) -> str:
    prompt = (
        f"<start_of_turn>user\n"
        f"{PROMPT_TEMPLATE.format(transcript=transcript, question=question)}"
        f"<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = out[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


sample_transcript = """
Ahmed: We need to finish the project report by Friday.
Sara: I will handle the design section.
John: I can review the budget numbers tonight.
Ahmed: Great. Let's meet again on Monday at 10 AM.
""".strip()

print("Q1:", ask(sample_transcript, "Who will handle the design section?"))
print("Q2:", ask(sample_transcript, "When is the next meeting?"))
print("Q3:", ask(sample_transcript, "What is the company IPO date?"))

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/gemma3-chatbot-lora"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
files.download(zip_path + ".zip")
print("Unzip to: chatbot/models/gemma3-chatbot-lora/")